In [1]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator as RGI, RectBivariateSpline as RBS
from skimage.registration import phase_cross_correlation as pcc
from skimage.transform import rotate
import astra
from numpy.fft import fftshift as SHFT
import matplotlib.pyplot as plt
import multiprocessing as mp
from multiprocessing.pool import ThreadPool
import tomosipo as ts
import functions
from skimage.restoration import estimate_sigma
import torch 
import functions
from scipy.ndimage import sobel, shift
import skimage.registration
from skimage.transform import warp_polar, rotate, rescale

In [3]:
num_of_projections= 2016
white = 8846
pixels_x = 2016
pixels_y = 1600

path = '/mnt/nas2/negin_data/Ti_Scans/Titanium_cylinder_WaygateM300/1/'
prefix = 'stepcylinder_'

# projections = read_log_data(path, prefix, white, num_of_projections, pixels_x, pixels_y)
# projection = projections[:, ::8, :]    #subsample the data
# np.save('Titanium_cylinder_Waygate_1.npy', projection)

projection = np.load('Titanium_cylinder_Waygate_1.npy')  # loading the subsampled version
# projection = projection[799:800, : , : ]    # taking fewer slices if needed

print(projections.shape)

plt.imshow(projections[:,1000,:])
plt.colorbar()

In [5]:
# scan parameters

n_rows = 1600
FOV = 2016
n_slices = 1600
detector_rows = 1600
detector_cols = 2016

voxel = 0.01773846
sod = 71.50000000
sdd = 806.15781689
rows = 1600
cols = 2016
pixel = 0.20
angles = projection.shape[1]

num_of_projections = projection.shape[1]
angles =  np.linspace(2*np.pi, 0, num = projection.shape[1], endpoint=False)
rot_step = -(angles[2]-angles[1])*180/np.pi


# geometry
det_x = - functions.shift_x_2d(projection, sod, sdd, pixel, rot_step)   # estimating the center of rotation misalignment
det_y = 0.0
eta = 0.0
theta = 0.0
phi = 0.0



In [ ]:
# setting up astra and FDK reconstruction

vectors = functions.vectors_astra(pixel, voxel, sod, sdd, rot_step, num_of_projections, det_x, det_y, eta, theta, phi)

proj_geom = astra.create_proj_geom('cone_vec', detector_rows, detector_cols, vectors)
vol_geom = astra.create_vol_geom(detector_cols, detector_rows, detector_cols)  
vol_id = astra.data3d.create('-vol', vol_geom)
proj_id = astra.data3d.create('-sino', proj_geom, projection)

reconstruction_id = astra.data3d.create('-vol', vol_geom)
alg_cfg = astra.astra_dict('FDK_CUDA')
alg_cfg['ProjectionDataId'] = proj_id
alg_cfg['ReconstructionDataId'] = reconstruction_id
algorithm_id = astra.algorithm.create(alg_cfg)
astra.algorithm.run(algorithm_id)
reconstruction = astra.data3d.get(reconstruction_id)

plt.imshow(reconstruction[:,:,0])
plt.colorbar()
plt.show()

# Regularized reconstruction
## smooth TV regularization

In [ ]:
device = 'cuda:0'

# defining everything in tomosipo, to work with gpu

vg = ts.astra.from_astra(vol_geom)
pg = ts.astra.from_astra(proj_geom)

A = ts.operator(vg, pg)

# estimating the noise level on the data
sigma = estimate_sigma(projection[projection.shape[0]//2-1:projection.shape[0]//2+1,:,:])

projection = torch.from_numpy(projection).to(device) # sending the projections to the gpu

In [ ]:
eps = torch.tensor(1e-7)
alpha = 1e-12
l_tv = 50000000
max_iter = 5
# x0 = torch.zeros(A.domain_shape).to(device)
x0 = torch.from_numpy(reconstruction).to(device)   # starting the regularized reconstruction from the FDK result, to lower the computation
rec_tv = functions.AGD(x0, A, projection, l_tv, alpha, max_iter, eps, sigma)
torch.cuda.empty_cache()

In [ ]:

plt.plot(reconstruction[1000,:,10])
plt.plot(rec_tv[1000,:,10].cpu())